<a href="https://colab.research.google.com/github/1900690/Water-meter-reading/blob/main/water_time_reading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ライブラリのインストール
!pip install moviepy

In [7]:
#@title 動画の切り取り設定（複数日・日中のみ自動抽出）
import os
import requests
import shutil
from moviepy.editor import VideoFileClip
from datetime import datetime, timedelta

# --- フォーム項目 ---
#@markdown ### ① 動画のURLリスト（1行に1つ）
video_urls_input = "https://github.com/1900690/image-movie-editing/releases/download/avi_isida2/20250806.AVI" #@param {type:"string"}

#@markdown ### ② 各動画の「1日目の開始時刻」リスト（1行に1つ）
start_times_input = "09:54:11" #@param {type:"string"}

#@markdown ---
#@markdown ### 撮影データの設定
capture_interval_sec = 600 #@param {type:"number"}

#@markdown ### 切り取りたい時間範囲（毎日この時間を抽出）
target_start_time_str = "06:00:00" #@param {type:"string"}
target_end_time_str = "18:00:00" #@param {type:"string"}
# ------------------

def parse_time(t_str):
    return datetime.strptime(t_str.strip(), "%H:%M:%S")

# 作業用フォルダのリセット
if os.path.exists("temp_clips"):
    shutil.rmtree("temp_clips")
os.makedirs("temp_clips", exist_ok=True)

# 保存用フォルダの作成
os.makedirs("daylight", exist_ok=True)

processed_clips = []
output_base_name = "output"

urls = [u.strip() for u in video_urls_input.strip().split('\n') if u.strip()]
start_times = [t.strip() for t in start_times_input.strip().split('\n') if t.strip()]

if urls:
    first_url_filename = urls[0].split('/')[-1]
    output_base_name = os.path.splitext(first_url_filename)[0]

for i, (url, start_wall_time) in enumerate(zip(urls, start_times)):
    base_date = datetime(2000, 1, 1)
    try:
        t_time = parse_time(start_wall_time)
        video_start_dt = base_date.replace(hour=t_time.hour, minute=t_time.minute, second=t_time.second)
    except Exception as e:
        print(f"❌ 時刻の形式が正しくありません ({start_wall_time}): {e}")
        continue

    print(f"\n--- Processing Video {i+1} ---")
    local_filename = f"source_{i}.avi"

    res = requests.get(url, stream=True)
    with open(local_filename, 'wb') as f:
        for chunk in res.iter_content(chunk_size=8192):
            f.write(chunk)

    try:
        with VideoFileClip(local_filename) as video:
            fps = video.fps
            total_real_seconds = video.duration * fps * capture_interval_sec
            video_end_dt = video_start_dt + timedelta(seconds=total_real_seconds)

            current_day = video_start_dt.replace(hour=0, minute=0, second=0)
            day_count = 0

            while current_day <= video_end_dt:
                t_start_base = parse_time(target_start_time_str)
                t_end_base = parse_time(target_end_time_str)

                day_target_start = current_day.replace(hour=t_start_base.hour, minute=t_start_base.minute, second=t_start_base.second)
                day_target_end = current_day.replace(hour=t_end_base.hour, minute=t_end_base.minute, second=t_end_base.second)

                overlap_start = max(video_start_dt, day_target_start)
                overlap_end = min(video_end_dt, day_target_end)

                if overlap_start < overlap_end:
                    diff_start = (overlap_start - video_start_dt).total_seconds()
                    diff_end = (overlap_end - video_start_dt).total_seconds()

                    v_start = diff_start / capture_interval_sec / fps
                    v_end = diff_end / capture_interval_sec / fps

                    print(f"  [Day {day_count+1}] Extracting: {overlap_start.strftime('%H:%M:%S')} - {overlap_end.strftime('%H:%M:%S')}")

                    clip = video.subclip(max(0, v_start), min(video.duration, v_end))
                    output_path = f"temp_clips/clip_v{i}_d{day_count}.mp4"
                    clip.write_videofile(output_path, codec="libx264", audio=False)
                    processed_clips.append(output_path)

                current_day += timedelta(days=1)
                day_count += 1

    except Exception as e:
        print(f"❌ エラー: {e}")

print(f"\n✅ 準備完了！保存先: daylight/{output_base_name}_daylight.mp4")


--- Processing Video 1 ---
  [Day 1] Extracting: 09:54:11 - 18:00:00
Moviepy - Building video temp_clips/clip_v0_d0.mp4.
Moviepy - Writing video temp_clips/clip_v0_d0.mp4



Moviepy - Done !
Moviepy - video ready temp_clips/clip_v0_d0.mp4
  [Day 2] Extracting: 06:00:00 - 18:00:00
Moviepy - Building video temp_clips/clip_v0_d1.mp4.
Moviepy - Writing video temp_clips/clip_v0_d1.mp4



Moviepy - Done !
Moviepy - video ready temp_clips/clip_v0_d1.mp4
  [Day 3] Extracting: 06:00:00 - 16:15:11
Moviepy - Building video temp_clips/clip_v0_d2.mp4.
Moviepy - Writing video temp_clips/clip_v0_d2.mp4



Moviepy - Done !
Moviepy - video ready temp_clips/clip_v0_d2.mp4

✅ 準備完了！保存先: daylight/20250806_daylight.mp4


In [8]:
#@title 動画の結合とdaylightフォルダへの保存
import os
from moviepy.editor import concatenate_videoclips
from google.colab import files

# ファイル名とパスの設定
final_filename = f"{output_base_name}_daylight.mp4"
final_path = os.path.join("daylight", final_filename)

if not processed_clips:
    print("エラー: 結合する動画がありません。")
else:
    print(f"「{final_path}」を生成中...")

    clips = [VideoFileClip(c) for c in processed_clips]
    final_video = concatenate_videoclips(clips)

    # daylightフォルダに書き出し
    final_video.write_videofile(final_path, codec="libx264")

    for c in clips:
        c.close()

    print(f"\n✅ 結合完了！ daylight フォルダに保存されました。")

    # ダウンロード
    #files.download(final_path)

「daylight/20250806_daylight.mp4」を生成中...
Moviepy - Building video daylight/20250806_daylight.mp4.
Moviepy - Writing video daylight/20250806_daylight.mp4



Moviepy - Done !
Moviepy - video ready daylight/20250806_daylight.mp4

✅ 結合完了！ daylight フォルダに保存されました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
#@title ① 動画の切り抜き設定（保存先：cut）
import os
from moviepy.editor import VideoFileClip

#@markdown ### 取得設定
#@markdown 処理したい動画ファイル、またはフォルダの**フルパス**を入力してください
target_full_path = "/content/daylight/20250806_daylight.mp4" #@param {type:"string"}

#@markdown ---
#@markdown ### 切り取り範囲（ピクセル）
#@markdown 左上の座標 (x1, y1) から 右下の座標 (x2, y2) を指定してください
x1 = 0 #@param {type:"number"}
y1 = 0 #@param {type:"number"}
x2 = 240 #@param {type:"number"}
y2 = 160 #@param {type:"number"}

# 保存用フォルダ名を "cut" に設定
output_folder = "cut"

# 保存用フォルダの作成
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"📁 保存先フォルダ「{output_folder}」を作成しました。")

# 処理対象のリストアップ
process_list = []

if os.path.isfile(target_full_path):
    # ファイル単体指定の場合
    process_list.append(target_full_path)
elif os.path.isdir(target_full_path):
    # フォルダ指定の場合
    video_extensions = ('.mp4', '.avi', '.mov', '.wmv', '.mkv')
    files = [os.path.join(target_full_path, f) for f in os.listdir(target_full_path) if f.lower().endswith(video_extensions)]
    process_list.extend(files)
else:
    print(" 指定されたパスが見つかりません。パスが正しいか再確認してください。")

# 実行処理
if not process_list:
    print(" 処理できる動画ファイルが見つかりませんでした。")
else:
    print(f" 合計 {len(process_list)} 件の処理を開始します...")

    for file_path in process_list:
        filename = os.path.basename(file_path)
        base_name = os.path.splitext(filename)[0]

        # 保存名の設定 (元のファイル名_cut.mp4)
        output_filename = f"{base_name}_cut.mp4"
        output_path = os.path.join(output_folder, output_filename)

        print(f"\n--- 処理中: {filename} ---")

        try:
            with VideoFileClip(file_path) as clip:
                # 指定範囲で切り取り (x1, y1, x2, y2)
                cropped_clip = clip.crop(x1=x1, y1=y1, x2=x2, y2=y2)

                # 書き出し（mp4形式、音声あり設定）
                cropped_clip.write_videofile(output_path, codec="libx264", audio=True)

            print(f"✅ 保存完了: {output_path}")

        except Exception as e:
            print(f"{filename} の処理中にエラーが発生しました: {e}")

print(f"\n すべての処理が終了しました。結果は「{output_folder}」フォルダを確認してください。")

📁 保存先フォルダ「cut」を作成しました。
🎬 合計 1 件の処理を開始します...

--- 処理中: 20250806_daylight.mp4 ---
Moviepy - Building video cut/20250806_daylight_cut.mp4.
Moviepy - Writing video cut/20250806_daylight_cut.mp4



Moviepy - Done !
Moviepy - video ready cut/20250806_daylight_cut.mp4
✅ 保存完了: cut/20250806_daylight_cut.mp4

✨ すべての処理が終了しました。結果は「cut」フォルダを確認してください。


In [ ]:
#@title ② クロップ済み動画の結合
import os
from moviepy.editor import VideoFileClip, concatenate_videoclips
from google.colab import files

#@markdown ### 結合設定
#@markdown 保存するファイル名を入力してください（拡張子は自動で.mp4になります）
final_output_name = "combined_video" #@param {type:"string"}

# フォルダ指定
input_folder = "cut"

# ファイルリストの取得とソート（名前順に結合するため）
if not os.path.exists(input_folder):
    print(f" 「{input_folder}」フォルダが見つかりません。先に切り抜き処理を行ってください。")
else:
    video_extensions = ('.mp4', '.avi', '.mov', '.wmv', '.mkv')
    video_files = sorted([os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.lower().endswith(video_extensions)])

    if not video_files:
        print(f" 「{input_folder}」フォルダ内に動画ファイルがありません。")
    else:
        print(f" {len(video_files)} 個の動画を結合しています...")

        clips = []
        try:
            # 各動画を読み込み
            for file_path in video_files:
                clip = VideoFileClip(file_path)
                clips.append(clip)

            # 動画を結合
            final_clip = concatenate_videoclips(clips)

            # 出力ファイル名の決定
            if not final_output_name.endswith(".mp4"):
                save_filename = f"{final_output_name}.mp4"
            else:
                save_filename = final_output_name

            # 書き出し
            print(f" 「{save_filename}」を書き出し中...")
            final_clip.write_videofile(save_filename, codec="libx264")

            # メモリ解放（重要）
            final_clip.close()
            for c in clips:
                c.close()

            print(f" 結合が完了しました！ダウンロードを開始します...")
            files.download(save_filename)

        except Exception as e:
            print(f" 結合処理中にエラーが発生しました: {e}")

In [11]:
#@title ③ 動画フレームの全フレーム切り出し
import os
from moviepy.editor import VideoFileClip

#@markdown ### 取得・保存設定
#@markdown 処理したい動画ファイルの**フルパス**を入力してください
target_video_path = "/content/daylight/20250806_daylight.mp4" #@param {type:"string"}

#@markdown フレーム画像を保存するフォルダ名
output_frames_folder = "cut_images" #@param {type:"string"}

#@markdown ---
#@markdown ### 切り出し範囲（時間：秒）
#@markdown 動画の何秒から何秒までを切り出すか指定してください
start_time_sec = 0.0 #@param {type:"number"}
end_time_sec = 1.0 #@param {type:"number"}

# 保存用フォルダの作成
if not os.path.exists(output_frames_folder):
    os.makedirs(output_frames_folder)
    print(f"📁 保存先フォルダ「{output_frames_folder}」を作成しました。")

# ファイルの存在確認
if not os.path.isfile(target_video_path):
    print(f"❌ 指定された動画ファイルが見つかりません: {target_video_path}")
else:
    filename = os.path.basename(target_video_path)
    base_name = os.path.splitext(filename)[0]

    try:
        with VideoFileClip(target_video_path) as clip:
            fps = clip.fps
            duration = clip.duration

            # 範囲の適正化
            s_time = max(0, start_time_sec)
            e_time = min(duration, end_time_sec)

            if s_time >= e_time:
                print(f"❌ 開始時間({s_time}秒)が終了時間({e_time}秒)以降です。")
            else:
                # 全フレームの時間を計算（1/fps秒ごと）
                frame_duration = 1.0 / fps
                current_time = s_time
                frame_index = 0

                print(f"🎬 動画FPS: {fps}")
                print(f"⏱ {s_time}秒から{e_time}秒までの全フレームを抽出します...")

                while current_time <= e_time:
                    # 連番と時間でファイル名作成
                    # (全フレーム抽出時は連番の方が管理しやすいため index を付与)
                    output_filename = f"{base_name}_{frame_index:05d}_{current_time:.3f}s.jpg"
                    output_path = os.path.join(output_frames_folder, output_filename)

                    # 保存
                    clip.save_frame(output_path, t=current_time)

                    # 1フレーム分時間を進める
                    current_time += frame_duration
                    frame_index += 1

                    # 100枚ごとに進捗表示
                    if frame_index % 100 == 0:
                        print(f"  {frame_index}枚目を処理中... ({current_time:.2f}秒)")

                print(f"\n✅ 完了！ 合計 {frame_index} 枚の画像を「{output_frames_folder}」に保存しました。")

    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")

🎬 動画FPS: 30.0
⏱ 0秒から1.0秒までの全フレームを抽出します...

✅ 完了！ 合計 31 枚の画像を「cut_images」に保存しました。
